In [1]:
import pandas as pd
import numpy as np

We train the `log_quadratic` model on the historical training dataset.

In [2]:
df_train = pd.read_parquet("output/historical_data/train/lev_sigma<0.6_std.parquet")

In [3]:
print("min vol", df_train["realized_vol"].min())
print("max vol", df_train["realized_vol"].max())

min vol 0.03185918243743643
max vol 0.5999720507676751


In [4]:
bins = np.linspace(df_train["realized_vol"].min(), df_train["realized_vol"].max(), 200)

df_train["realized_vol"] = pd.cut(x=df_train["realized_vol"], bins=bins)

In [5]:
df_train["realized_vol"] = df_train["realized_vol"].apply(lambda x: x.mid)

In [ ]:
grouped_df_train = df_train.groupby(by=["leverage_factor", "realized_vol"])[
    ["PnL"]
].std()

/tmp/ipykernel_25314/3578563250.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_df_train = df_train.groupby(by=["leverage_factor", "realized_vol"])[["PnL"]].std()


In [7]:
grouped_df_train.reset_index(inplace=True)

In [8]:
grouped_df_train.rename(columns={"PnL": "PnL_std"}, inplace=True)

In [9]:
grouped_df_train

Ticker,leverage_factor,realized_vol,PnL_std
0,1.01,0.03330,0.000003
1,1.01,0.03615,0.000029
2,1.01,0.03900,0.000065
3,1.01,0.04185,0.000084
4,1.01,0.04470,0.000100
...,...,...,...
19895,3.00,0.58750,0.186795
19896,3.00,0.59000,0.146202
19897,3.00,0.59250,0.128614
19898,3.00,0.59550,0.146250


In [ ]:
import models

model = models.generate_model(degree=2, take_log=True)
model.fit(
    grouped_df_train[["leverage_factor", "realized_vol"]], grouped_df_train["PnL_std"]
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('poly', ...), ('log_linear', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",2
,"interaction_only interaction_only: bool, default=FalseIf `True`, only interaction features are produced: features that areproducts of at most `degree` *distinct* input features, i.e. terms withpower of 2 or higher of the same input feature are excluded:- included: `x[0]`, `x[1]`, `x[0] * x[1]`, etc.- excluded: `x[0] ** 2`, `x[0] ** 2 * x[1]`, etc.",False
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a linear model).",True
,"order order: {'C', 'F'}, default='C'Order of output array in the dense case. `'F'` order is faster tocompute, but may slow down subsequent estimators... versionadded:: 0.21",'C'
,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",LinearRegress...tercept=False)
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also 

Test

In [ ]:
from sklearn.metrics import mean_squared_error as mse

df_test = pd.read_parquet("output/historical_data/train/lev_sigma<0.6_std.parquet")
df_test["realized_vol"] = pd.cut(x=df_test["realized_vol"], bins=bins).apply(
    lambda x: x.mid
)

grouped_df_test = df_test.groupby(by=["leverage_factor", "realized_vol"])[["PnL"]].std()
grouped_df_test.reset_index(inplace=True)
grouped_df_test.rename(columns={"PnL": "PnL_std"}, inplace=True)

grouped_df_test

/tmp/ipykernel_25314/4151807085.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_df_test = df_test.groupby(by=["leverage_factor", "realized_vol"])[["PnL"]].std()


Ticker,leverage_factor,realized_vol,PnL_std
0,1.01,0.03330,0.000003
1,1.01,0.03615,0.000029
2,1.01,0.03900,0.000065
3,1.01,0.04185,0.000084
4,1.01,0.04470,0.000100
...,...,...,...
19895,3.00,0.58750,0.186795
19896,3.00,0.59000,0.146202
19897,3.00,0.59250,0.128614
19898,3.00,0.59550,0.146250


In [ ]:
pred = model.predict(grouped_df_test[["leverage_factor", "realized_vol"]])

print("mse =", mse(grouped_df_test["PnL_std"], pred))
print(
    "R2  =",
    model.score(
        grouped_df_test[["leverage_factor", "realized_vol"]], grouped_df_test["PnL_std"]
    ),
)

mse = 0.00023794036830907588
R2  = 0.8637408476355051
